In [2]:
from physys.loader import load_config
from physys.runtime import PHYSys

# 1. Load the configuration using the correct loader function
config = load_config("config.json") 

# 2. Flip the return_channel flag on the time waveform configuration
config.waveforms.time.return_channel = True

# 3. Instantiate the simulation orchestrator
sim = PHYSys(config)

In [3]:
from physys.loader import load_config
from physys.runtime import PHYSys

config = load_config("config.json")
config.waveforms.time.return_channel = True

sim = PHYSys(config)

bits, llr, x, y = sim.generate_iq(batch_size=4, num_symbols=1024, ebno_db=10.0)

h_time = sim._last_h_time
print(type(h_time))
print(h_time.shape)

<class 'torch.Tensor'>
torch.Size([4, 1, 1, 1, 1, 1039, 16])


In [4]:
import torch

h = sim._last_h_time  # [4, 1, 1, 1, 1, 1039, 16]
h = h.squeeze()        # -> [4, 1039, 16], drops the singleton rx/tx dims

mag = h.abs()                      # [4, 1039, 16]
per_time_step = mag.mean(dim=-1)   # [4, 1039] -- avg tap energy per time step

for b in range(h.shape[0]):
    row = per_time_step[b]
    print(f"batch {b}: mean={row.mean():.6f} std={row.std():.6f} "
          f"min={row.min():.6f} max={row.max():.6f} "
          f"rel_std={(row.std()/row.mean()):.4%}")

batch 0: mean=0.000000 std=0.000000 min=0.000000 max=0.000000 rel_std=0.0000%
batch 1: mean=0.000000 std=0.000000 min=0.000000 max=0.000000 rel_std=0.0000%
batch 2: mean=0.000000 std=0.000000 min=0.000000 max=0.000000 rel_std=0.0000%
batch 3: mean=0.000000 std=0.000000 min=0.000000 max=0.000000 rel_std=0.0000%


In [5]:
print(h.dtype)
print("global max |h|:", h.abs().max().item())
print("global min |h|:", h.abs().min().item())
print("any nan:", torch.isnan(h.abs()).any().item())
print("any inf:", torch.isinf(h.abs()).any().item())

# print a few raw tap values, batch 0, first and last time step, full precision
print(h[0, 0, :])
print(h[0, -1, :])

torch.complex64
global max |h|: 1.1168627906954498e-06
global min |h|: 6.819489417608793e-09
any nan: False
any inf: False
tensor([ 1.7454e-08-4.5538e-08j, -2.0817e-08+5.4229e-08j,
         2.5778e-08-6.6995e-08j, -3.3816e-08+8.7558e-08j,
         4.9027e-08-1.2609e-07j, -8.8146e-08+2.2362e-07j,
         4.9231e-07-1.0025e-06j,  1.2356e-07-2.6408e-07j,
         7.1010e-08+3.4944e-07j,  4.6210e-08-9.4466e-08j,
        -2.8905e-08+7.1585e-08j,  2.2506e-08-5.7637e-08j,
        -1.8553e-08+4.8062e-08j,  1.5807e-08-4.1162e-08j,
        -1.3777e-08+3.5974e-08j,  1.2211e-08-3.1937e-08j], device='cuda:0')
tensor([ 1.7454e-08-4.5538e-08j, -2.0817e-08+5.4229e-08j,
         2.5778e-08-6.6995e-08j, -3.3816e-08+8.7558e-08j,
         4.9027e-08-1.2609e-07j, -8.8146e-08+2.2362e-07j,
         4.9231e-07-1.0025e-06j,  1.2356e-07-2.6408e-07j,
         7.1010e-08+3.4944e-07j,  4.6210e-08-9.4466e-08j,
        -2.8905e-08+7.1585e-08j,  2.2506e-08-5.7637e-08j,
        -1.8553e-08+4.8062e-08j,  1.5807e-08-4.

The two time-step vectors you just printed are identical to displayed precision, and the magnitudes are just path-loss scale (~1e-8 to ~1e-6), which is why the mean/std collapsed to 0.000000 before — not a bug, just a precision-display issue in the earlier script.

Answer to step 1: the channel is static within a burst. Makes sense given what you found earlier — SystemLevelChannelConfig has no mobility field, so gen_single_sector_topology() places a stationary UE, and Sionna's system-level channel model doesn't evolve h over a 1024-symbol window without a speed parameter driving Doppler.

That means:

equalize() needs one FFT'd H per burst, not sub-block segmentation. You can take any single time slice, e.g. h_time[..., 0, :] (16-tap vector), since all 1039 are equal.
Scope stays exactly as planned — no added complexity from time-variation.

In [7]:
from physys.loader import load_config
from physys.runtime import PHYSys

config = load_config("config.json")
config.waveforms.time.return_channel = True
config.channels.system_level.enable_pathloss = False
config.channels.system_level.enable_shadow_fading = False

sim = PHYSys(config)
bits, llr, x, y = sim.generate_iq(batch_size=4, num_symbols=1024, ebno_db=10.0)

h = sim._last_h_time.squeeze()
print("global max |h|:", h.abs().max().item())
print("global min |h|:", h.abs().min().item())

global max |h|: 1.3412673473358154
global min |h|: 0.006312169134616852


### synthetic unit test

In [8]:
import torch

def equalize(y_time, h_time, no, num_time_samples):
    """
    SC-FDE style genie-aided equalizer.
    y_time:  [..., num_time_samples] complex, time-domain rx signal (TimeChannel output, untruncated)
    h_time:  [..., l_tot] complex, per-burst channel tap vector (assumes time-constant within burst)
    no:      scalar noise variance (linear, not dB)
    num_time_samples: length of y_time's last axis (1039 in our case, NOT vector_len)

    Returns x_hat: [..., num_time_samples] complex, equalized time-domain signal
    (caller truncates to vector_len afterward, same as the existing non-equalized path)
    """
    l_tot = h_time.shape[-1]
    fft_len = num_time_samples  # zero-pad h out to this length, NOT vector_len

    Y = torch.fft.fft(y_time, n=fft_len, dim=-1)
    H = torch.fft.fft(h_time, n=fft_len, dim=-1)  # zero-pads h_time's l_tot taps to fft_len

    H_conj = torch.conj(H)
    H_mag_sq = (H * H_conj).real

    X_hat = Y * H_conj / (H_mag_sq + no)  # MMSE regularization by noise variance

    x_hat = torch.fft.ifft(X_hat, n=fft_len, dim=-1)
    return x_hat


# --- synthetic unit test ---
torch.manual_seed(0)

fft_len = 1039
l_tot = 16

x_true = (torch.randn(fft_len) + 1j*torch.randn(fft_len)) / (2**0.5)   # random symbols, unit power
h_true = (torch.randn(l_tot) + 1j*torch.randn(l_tot)) * 0.3            # random tap vector, similar scale to what we measured

# linear convolution, same as TimeChannel produces (output longer than input)
y_full = torch.zeros(fft_len + l_tot - 1, dtype=torch.complex64)
for k in range(l_tot):
    y_full[k:k+fft_len] += h_true[k] * x_true

# TimeChannel's actual output length convention: input + l_tot - 1
# so num_time_samples in this synthetic test = len(y_full), matching the real 1039-vs-1024 pattern
no = torch.tensor(1e-6)  # negligible noise, to check near-exact recovery first

x_hat = equalize(y_full, h_true, no, num_time_samples=y_full.shape[-1])
x_hat_truncated = x_hat[:fft_len]  # what _apply_time_channel would keep

err = (x_hat_truncated - x_true).abs()
print("max abs error:", err.max().item())
print("mean abs error:", err.mean().item())

max abs error: 2.8243870474398136e-05
mean abs error: 1.1314285984553862e-05


In [9]:
# realistic noise level -- pick something near what a low-SNR AMR point would see.
# With |h| up to ~1.34 and modulation/SNR sweeping down to -10dB, no can be
# meaningfully larger than the negligible-noise case above.
for no_val in [1e-6, 1e-4, 1e-3, 1e-2, 1e-1]:
    no = torch.tensor(no_val)
    x_hat = equalize(y_full, h_true, no, num_time_samples=y_full.shape[-1])
    x_hat_truncated = x_hat[:fft_len]
    err = (x_hat_truncated - x_true).abs()
    print(f"no={no_val:.0e}  max_err={err.max().item():.4f}  mean_err={err.mean().item():.4f}")

no=1e-06  max_err=0.0000  mean_err=0.0000
no=1e-04  max_err=0.0028  mean_err=0.0011
no=1e-03  max_err=0.0257  mean_err=0.0104
no=1e-02  max_err=0.1633  mean_err=0.0634
no=1e-01  max_err=0.5900  mean_err=0.1868
